# Testing for Late Fusion Techniques

### Here we aim to combine the discriminative powers of:

1. Structural Features (FUSE embeddings)
2. Semantic Features (Word-2-Vec) [provided in the OGBN benchmark dataset]

### Two techniques were tested:

1. Simple Concatenation
2. Contrastive Alignment (CLIP-based)

## Results Summary

The table below reports the average classification accuracies obtained using the two fusion strategies: simple concatenation and CLIP-based contrastive alignment. Each fused representation was evaluated using two downstream classifiers: a Multi-Layer Perceptron (MLP) and Multinomial Logistic Regression.

| Fusion Technique | MLP Average Accuracy | Multinomial Logistic Regression Average Accuracy |
|---|---:|---:|
| Simple Concatenation | `0.7047` | `0.5759` |
| CLIP-based Contrastive Alignment | `0.6821` | `0.7632` |

## Interpretation


- Simple concatenation directly combines the two feature types without explicitly aligning their embedding geometries. Therefore, it may require a more expressive nonlinear classifier such as an MLP to extract useful interactions between structural and semantic information.

- In contrast, CLIP-based contrastive alignment aims to learn a shared representation space where the two feature types are geometrically aligned. If this learned representation is effective, it should become more linearly separable, making multinomial logistic regression a strong benchmark for evaluating the quality of the learned embedding space.

Thus, better performance with multinomial logistic regression indicates that the CLIP-based fusion method produces a cleaner and more discriminative embedding geometry.

In [ ]:
!pip install ogb
!pip install torch_geometric

import torch
import numpy as np
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import to_torch_csr_tensor   

import torch
from ogb.nodeproppred import NodePropPredDataset 


import wandb 
from sklearn.linear_model import LogisticRegression

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [2]:

original_torch_load = torch.load

def patched_torch_load(*args, **kwargs):
    kwargs["weights_only"] = False
    return original_torch_load(*args, **kwargs)

torch.load = patched_torch_load

In [ ]:
# loading the pre-saved dataset indices:

train_idx = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_train_index")
test_idx= torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_test_index")
valid_idx = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_valid_index")
labels_tensor = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_labels_tensor")
num_classes = 47
node_features = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-metadataokayy/Products_ogbn_node_features")

torch.cuda.empty_cache()

## Testing with Simple Concatenation:

In [4]:
S = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/s-products-normed-k100/S_products_normed_k100")
S_given = node_features

In [5]:
node_features.shape

torch.Size([2449029, 100])

### MLP:

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import TensorDataset, DataLoader

S = S.to(device)
S_given = S_given.to(device)

S_generated = torch.cat((S, S_given) , dim = 1)
del S, S_given

print(f"Shape of concatenated embeddings: {S_generated.shape}")

train_idx = torch.tensor(train_idx).to(device)
valid_idx =  torch.tensor(valid_idx).to(device)
test_idx  =  torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device) 


S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test  = S_generated[test_idx].detach()
y_test  = labels_tensor[test_idx].long()
del S_generated
print(f"Train nodes: {S_train.shape[0]} | Valid nodes: {S_valid.shape[0]} | Test nodes: {S_test.shape[0]}")

mean = S_train.mean(dim=0, keepdim=True)
std  = S_train.std(dim=0, keepdim=True) + 1e-7

S_train_scaled = (S_train - mean) / std
S_valid_scaled = (S_valid - mean) / std
S_test_scaled  = (S_test - mean) / std

batch_size = 2048
train_dataset = TensorDataset(S_train_scaled, y_train)
train_loader  = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden), 
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(hidden, hidden), 
            nn.ReLU(),
            nn.Dropout(0.5), 
            nn.Linear(hidden, out_dim)
        )
    def forward(self, x): return self.net(x)

K = S_train.shape[1]
model = MLP(K, 512, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 200
losses = []
best_val_acc = 0.0
best_model_weights = None
patience = 20  
epochs_no_improve = 0
target_threshold = 0.99

for epoch in range(epochs):
    
    model.train()
    epoch_loss = 0.0
    
    for batch_x, batch_y in train_loader:
        optimizer.zero_grad()
        logits = model(batch_x)
        loss = criterion(logits, batch_y)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()
        
    avg_train_loss = epoch_loss / len(train_loader)
    losses.append(epoch_loss)
    
    model.eval()
    with torch.no_grad():
        val_logits = model(S_valid_scaled)
        val_preds = val_logits.argmax(dim=1)
        val_correct = (val_preds == y_valid).sum().item()
        val_acc = val_correct / len(y_valid)

    print(f'Epoch {epoch+1:3d}/{epochs} | Train Loss: {avg_train_loss:.4f} | Val Acc: {val_acc:.4f}')

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(model.state_dict())
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    if val_acc >= target_threshold:
        print(f"\nTarget Validation Accuracy ({target_threshold}) reached! Stopping early.")
        break
        
    if epochs_no_improve >= patience:
        print(f"\nEarly stopping triggered! No improvement for {patience} epochs.")
        break

print(f"\nLoading best model weights (Val Acc: {best_val_acc:.4f}) for Final Testing...")
model.load_state_dict(best_model_weights)


del S_train, S_valid

test_loader = DataLoader(
    TensorDataset(S_test_scaled, y_test),
    batch_size=4096,
    shuffle=False
)

model.eval()

all_preds = []

with torch.no_grad():
    for batch_x, _ in test_loader:
        batch_x = batch_x.to(device)
        logits = model(batch_x)
        preds = logits.argmax(dim=1)
        all_preds.append(preds.cpu())

test_preds = torch.cat(all_preds)
test_correct = (test_preds == y_test.cpu()).sum().item()
test_acc = test_correct / len(y_test)

print(f"Final Test Accuracy: {test_acc:.4f}")

Shape of concatenated embeddings: torch.Size([2449029, 200])


/tmp/ipykernel_57/3965087859.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  labels_tensor = torch.tensor(labels_tensor).to(device)


Train nodes: 196615 | Valid nodes: 39323 | Test nodes: 2213091
Epoch   1/200 | Train Loss: 0.3506 | Val Acc: 0.8582
Epoch   2/200 | Train Loss: 0.0082 | Val Acc: 0.8571
Epoch   3/200 | Train Loss: 0.0033 | Val Acc: 0.8563
Epoch   4/200 | Train Loss: 0.0015 | Val Acc: 0.8556
Epoch   5/200 | Train Loss: 0.0010 | Val Acc: 0.8536
Epoch   6/200 | Train Loss: 0.0007 | Val Acc: 0.8548
Epoch   7/200 | Train Loss: 0.0005 | Val Acc: 0.8541
Epoch   8/200 | Train Loss: 0.0004 | Val Acc: 0.8543
Epoch   9/200 | Train Loss: 0.0009 | Val Acc: 0.8511
Epoch  10/200 | Train Loss: 0.0004 | Val Acc: 0.8537
Epoch  11/200 | Train Loss: 0.0005 | Val Acc: 0.8504
Epoch  12/200 | Train Loss: 0.0006 | Val Acc: 0.8466
Epoch  13/200 | Train Loss: 0.0005 | Val Acc: 0.8537
Epoch  14/200 | Train Loss: 0.0004 | Val Acc: 0.8471
Epoch  15/200 | Train Loss: 0.0005 | Val Acc: 0.8478
Epoch  16/200 | Train Loss: 0.0005 | Val Acc: 0.8487
Epoch  17/200 | Train Loss: 0.0005 | Val Acc: 0.8511
Epoch  18/200 | Train Loss: 0.0007 |

### Multinomial Logistic Regressiom:

In [7]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score

S = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/s-products-normed-k100/S_products_normed_k100")
S_given = node_features

S = S.to(device)
S_given = S_given.to(device)

S_generated = torch.cat((S, S_given) , dim = 1)
del S, S_given

print(f"Shape of concatenated embeddings: {S_generated.shape}")

train_idx = torch.tensor(train_idx).to(device)
valid_idx =  torch.tensor(valid_idx).to(device)
test_idx  =  torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device) 


S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test  = S_generated[test_idx].detach()
y_test  = labels_tensor[test_idx].long()



print("\n--- Transferring Data to CPU for Scikit-Learn ---")
X_train_np = S_train.cpu().numpy()
y_train_np = y_train.cpu().numpy()

X_test_np  = S_test.cpu().numpy()
y_test_np  = y_test.cpu().numpy()

print("Training Logistic Regression...")
clf = LogisticRegression(max_iter=1000, multi_class='multinomial', n_jobs=-1)
clf.fit(X_train_np, y_train_np)

print("Evaluating on Test Set...")
y_pred = clf.predict(X_test_np)


print(f"Accuracy: {accuracy_score(y_pred,y_test_np )}")


print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test_np, y_pred, digits=4))

Shape of concatenated embeddings: torch.Size([2449029, 200])

--- Transferring Data to CPU for Scikit-Learn ---


/tmp/ipykernel_57/311732516.py:15: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx).to(device)
/tmp/ipykernel_57/311732516.py:16: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid_idx =  torch.tensor(valid_idx).to(device)
/tmp/ipykernel_57/311732516.py:17: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx  =  torch.tensor(test_idx).to(device)
/tmp/ipykernel_57/311732516.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach(

Training Logistic Regression...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluating on Test Set...
Accuracy: 0.47595512339980595

--- Logistic Regression Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.3260    0.3941    0.3568    100670
           1     0.3898    0.3646    0.3768     95236
           2     0.6777    0.6543    0.6658    102870
           3     0.3162    0.4077    0.3562    135255
           4     0.4942    0.8459    0.6239    596195
           5     0.4077    0.2839    0.3347     35985
           6     0.3387    0.4301    0.3790    140356
           7     0.6036    0.5146    0.5555    152202
           8     0.6769    0.7558    0.7142     97679
           9     0.6811    0.4855    0.5669     59852
          10     0.3951    0.3361    0.3633     47165
          11     0.2239    0.0921    0.1306     28933
          12     0.5607    0.1987    0.2934    128632
          13     0.4300    0.2783    0.3379     88694
          14     0.3355    0.0757    0.1235      2734
          15     0.5745    0.2073    0.3046     23871
          16     0.7207    0.0251    0.0485     83019
          17     0.6571    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


## CLIP based Contrastive Alignment :

### Contrastively Fused embedding generation:

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import copy
import wandb 
from torch.utils.data import DataLoader, TensorDataset

# ==========================================================
#  Dual Projection (Encoding the Space)
# ==========================================================
class ContrastiveProjectionHead(nn.Module):
    def __init__(self, input_dim=100, hidden_dim=256, latent_dim=128):
        super().__init__()
        self.proj_str = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        self.proj_sem = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.BatchNorm1d(hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, latent_dim)
        )
        
        self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))

    def forward(self, x_str, x_sem):
        z_str = self.proj_str(x_str)
        z_sem = self.proj_sem(x_sem)

        z_str = F.normalize(z_str, p=2, dim=-1)
        z_sem = F.normalize(z_sem, p=2, dim=-1)

        return z_str, z_sem

# ==========================================================
#  InfoNCE Loss Function
# ==========================================================
def symmetric_infonce_loss(z_str, z_sem, logit_scale):
    logits_per_str = logit_scale.exp() * torch.matmul(z_str, z_sem.t())
    logits_per_sem = logits_per_str.t() 

    batch_size = z_str.shape[0]
    labels = torch.arange(batch_size, dtype=torch.long, device=z_str.device)

    loss_str = F.cross_entropy(logits_per_str, labels)
    loss_sem = F.cross_entropy(logits_per_sem, labels)

    return (loss_str + loss_sem) / 2

S = S.to(device)
S_given = S_given.to(device)

print("\n--- Standardizing Tensors ---")
mean_S = S.mean(dim=0, keepdim=True)
std_S = S.std(dim=0, keepdim=True) + 1e-7
S_scaled = (S - mean_S) / std_S

mean_S_given = S_given.mean(dim=0, keepdim=True)
std_S_given = S_given.std(dim=0, keepdim=True) + 1e-7
S_given_scaled = (S_given - mean_S_given) / std_S_given

batch_size = 4096 
dataset = TensorDataset(S_scaled, S_given_scaled) 
loader = DataLoader(dataset, batch_size=batch_size, shuffle=True, drop_last=True)

# ==========================================================
# Training Loop
# ==========================================================
config = {
    "input_dim": 100,
    "hidden_dim": 512,
    "latent_dim": 128,
    "batch_size": batch_size,
    "learning_rate": 0.005,
    "weight_decay": 1e-4,
    "epochs": 1000,
    "patience": 10
}

wandb.init(
    project="fuse-ogbn-products-contrastive",
    name="latent-alignment-infonce",
    config=config
)

model = ContrastiveProjectionHead(
    input_dim=config["input_dim"], 
    hidden_dim=config["hidden_dim"], 
    latent_dim=config["latent_dim"]
).to(device)

optimizer = optim.Adam(model.parameters(), lr=config["learning_rate"], weight_decay=config["weight_decay"])
scheduler = optim.lr_scheduler.StepLR(optimizer, step_size=20, gamma=0.9)

print("\n--- Starting Contrastive Latent Alignment ---")

best_loss = float('inf')
epochs_no_improve = 0
best_model_weights = copy.deepcopy(model.state_dict())

for epoch in range(config["epochs"]):
    model.train()
    total_loss = 0.0
    
    total_top1_acc = 0.0
    total_pos_sim = 0.0
    total_neg_sim = 0.0
    
    for batch_str, batch_sem in loader:
        optimizer.zero_grad()
        
        z_str, z_sem = model(batch_str, batch_sem)
        loss = symmetric_infonce_loss(z_str, z_sem, model.logit_scale)
        
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        
        with torch.no_grad():
            raw_sim = torch.matmul(z_str, z_sem.t())
            curr_batch_size = raw_sim.shape[0]
            labels = torch.arange(curr_batch_size, device=raw_sim.device)
            
            preds = raw_sim.argmax(dim=1)
            acc = (preds == labels).float().mean().item()
            total_top1_acc += acc
            
            pos_sim = torch.diag(raw_sim).mean().item()
            total_pos_sim += pos_sim
            
            sum_all = raw_sim.sum().item()
            sum_pos = torch.diag(raw_sim).sum().item()
            neg_sim = (sum_all - sum_pos) / (curr_batch_size * (curr_batch_size - 1))
            total_neg_sim += neg_sim
            
    
    num_batches = len(loader)
    avg_loss = total_loss / num_batches
    avg_top1 = total_top1_acc / num_batches
    avg_pos = total_pos_sim / num_batches
    avg_neg = total_neg_sim / num_batches
    
    current_temp = model.logit_scale.exp().item()
    
    scheduler.step()

    wandb.log({
        "epoch": epoch + 1,
        "InfoNCE_Loss": avg_loss,
        "Top1_Accuracy_Pct": avg_top1 * 100,
        "Pos_Similarity": avg_pos,
        "Neg_Similarity": avg_neg,
        "Temp_Scale": current_temp,
        "Learning_Rate": optimizer.param_groups[0]['lr']
    })

    if avg_loss < best_loss - 1e-3:
        best_loss = avg_loss
        epochs_no_improve = 0
        best_model_weights = copy.deepcopy(model.state_dict())
    else:
        epochs_no_improve += 1

    if (epoch + 1) % 1 == 0 or epoch == 0:
        print(f"Epoch {epoch+1:3d}/{config['epochs']} | Loss: {avg_loss:.4f} | "
              f"Top-1 Acc: {avg_top1*100:.1f}% | "
              f"Pos Sim: {avg_pos:.3f} | Neg Sim: {avg_neg:.3f} | "
              f"Temp: {current_temp:.1f}")



    if epoch == 50:
        break
    if epochs_no_improve >= config["patience"]:
        print(f"\nEarly stopping triggered! No improvement in loss for {config['patience']} epochs.")
        break

print("Contrastive Alignment Complete.")

# 
print(f"\nLoading best model weights (Loss: {best_loss:.4f}) for extraction...")
model.load_state_dict(best_model_weights)
model.eval()

final_dataset = TensorDataset(S_scaled, S_given_scaled)
final_loader = DataLoader(final_dataset, batch_size=4096, shuffle=False)

all_z_str = []
all_z_sem = []

with torch.no_grad():
    for batch_str, batch_sem in final_loader:
        batch_z_str, batch_z_sem = model(batch_str, batch_sem)

        all_z_str.append(batch_z_str.cpu())
        all_z_sem.append(batch_z_sem.cpu())

Final_Z_str = torch.cat(all_z_str, dim=0)
Final_Z_sem = torch.cat(all_z_sem, dim=0)

Z_fused = Final_Z_str + Final_Z_sem

print(f"Final Fused Embedding Matrix Shape: {Z_fused.shape}")

# Finish W&B run
wandb.finish()


--- Standardizing Tensors ---


/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mehul22 (mehul22-iiser-thiruvananthapuram) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin



--- Starting Contrastive Latent Alignment ---
Epoch   1/1000 | Loss: 6.5343 | Top-1 Acc: 2.4% | Pos Sim: 0.496 | Neg Sim: 0.263 | Temp: 19.7
Epoch   2/1000 | Loss: 6.1920 | Top-1 Acc: 3.8% | Pos Sim: 0.448 | Neg Sim: 0.235 | Temp: 21.9
Epoch   3/1000 | Loss: 6.0992 | Top-1 Acc: 4.4% | Pos Sim: 0.416 | Neg Sim: 0.213 | Temp: 23.7
Epoch   4/1000 | Loss: 6.0522 | Top-1 Acc: 4.7% | Pos Sim: 0.396 | Neg Sim: 0.199 | Temp: 24.5
Epoch   5/1000 | Loss: 6.0209 | Top-1 Acc: 5.0% | Pos Sim: 0.384 | Neg Sim: 0.190 | Temp: 24.9
Epoch   9/1000 | Loss: 5.9621 | Top-1 Acc: 5.4% | Pos Sim: 0.364 | Neg Sim: 0.174 | Temp: 25.7
Epoch  10/1000 | Loss: 5.9573 | Top-1 Acc: 5.4% | Pos Sim: 0.362 | Neg Sim: 0.172 | Temp: 25.6
Epoch  11/1000 | Loss: 5.9520 | Top-1 Acc: 5.5% | Pos Sim: 0.361 | Neg Sim: 0.171 | Temp: 25.6
Epoch  12/1000 | Loss: 5.9476 | Top-1 Acc: 5.5% | Pos Sim: 0.360 | Neg Sim: 0.169 | Temp: 25.8
Epoch  13/1000 | Loss: 5.9446 | Top-1 Acc: 5.6% | Pos Sim: 0.359 | Neg Sim: 0.169 | Temp: 25.9
Epo

InfoNCE_Loss,█▄▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Learning_Rate,████████████████▄▄▄▄▄▄▄▄▄▄▄▄▄▄▄▁▁▁▁▁▁▁▁▁
Neg_Similarity,█▆▅▄▃▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Pos_Similarity,█▆▄▃▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
Temp_Scale,▁▃▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇▇█▇██▇▇▇▇█▇█████████
Top1_Accuracy_Pct,▁▄▅▆▆▇▇▇▇▇▇▇▇▇▇▇▇███████████████████████
epoch,▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▇▇▇▇▇▇███
InfoNCE_Loss,5.88784
Learning_Rate,0.00405
Neg_Similarity,0.15945
Pos_Similarity,0.34585


In [9]:

torch.save(Z_fused, "Clip_Products.pth")


In [8]:
clip_fused = torch.load(r"/kaggle/input/datasets/mehulgoyal1729/ogbn-products-clip-fused/Clip_Products.pth")

In [9]:
clip_fused.shape

torch.Size([2449029, 128])

### MLP

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import copy
from torch.utils.data import TensorDataset, DataLoader

S_generated = clip_fused

print(f"Shape of concatenated embeddings: {S_generated.shape}")

train_idx = torch.tensor(train_idx).to(device)
valid_idx = torch.tensor(valid_idx).to(device)
test_idx  = torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device)
S_generated = S_generated.to(device)



print(S_generated.device)
print(train_idx.device)

S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test = S_generated[test_idx].detach()
y_test = labels_tensor[test_idx].long()

del S_generated

print(
    f"Train nodes: {S_train.shape[0]} | "
    f"Valid nodes: {S_valid.shape[0]} | "
    f"Test nodes: {S_test.shape[0]}"
)

mean = S_train.mean(dim=0, keepdim=True)
std = S_train.std(dim=0, keepdim=True) + 1e-7

S_train_scaled = (S_train - mean) / std
S_valid_scaled = (S_valid - mean) / std
S_test_scaled  = (S_test - mean) / std

batch_size = 2048

train_dataset = TensorDataset(S_train_scaled, y_train)
train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True
)

class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, hidden),
            nn.BatchNorm1d(hidden),   
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden, hidden),
            nn.BatchNorm1d(hidden),  
            nn.ReLU(),
            nn.Dropout(0.5),

            nn.Linear(hidden, out_dim)
        )

    def forward(self, x):
        return self.net(x)

K = S_train.shape[1]

model = MLP(K, 512, num_classes).to(device)

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

epochs = 200
best_val_acc = 0
best_model_weights = None

patience = 20
epochs_no_improve = 0
target_threshold = 0.99

for epoch in range(epochs):


    model.train()

    epoch_loss = 0

    for batch_x, batch_y in train_loader:

        optimizer.zero_grad()

        logits = model(batch_x)

        loss = criterion(logits, batch_y)

        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    avg_train_loss = epoch_loss / len(train_loader)


    model.eval()

    with torch.no_grad():

        train_logits = model(S_train_scaled)
        train_preds = train_logits.argmax(dim=1)

        train_correct = (
            train_preds == y_train
        ).sum().item()

        train_acc = train_correct / len(y_train)

        val_logits = model(S_valid_scaled)

        val_preds = val_logits.argmax(dim=1)

        val_correct = (
            val_preds == y_valid
        ).sum().item()

        val_acc = val_correct / len(y_valid)

    gap = train_acc - val_acc

    print(
        f"Epoch {epoch+1:3d}/{epochs} | "
        f"Loss: {avg_train_loss:.4f} | "
        f"Train Acc: {train_acc:.4f} | "
        f"Val Acc: {val_acc:.4f} | "
        f"Gap: {gap:.4f}"
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        best_model_weights = copy.deepcopy(
            model.state_dict()
        )
        epochs_no_improve = 0

    else:
        epochs_no_improve += 1

    if val_acc >= target_threshold:
        print(
            f"\nTarget validation accuracy "
            f"{target_threshold} reached."
        )
        break

    if epochs_no_improve >= patience:
        print(
            f"\nEarly stopping "
            f"(no improvement for {patience} epochs)"
        )
        break

print(
    f"\nLoading best model "
    f"(Val Acc={best_val_acc:.4f})"
)

model.load_state_dict(best_model_weights)

test_loader = DataLoader(
    TensorDataset(S_test_scaled, y_test),
    batch_size=4096,
    shuffle=False
)

model.eval()

all_preds = []

with torch.no_grad():

    for batch_x, _ in test_loader:

        logits = model(batch_x)

        preds = logits.argmax(dim=1)

        all_preds.append(
            preds.cpu()
        )

test_preds = torch.cat(all_preds)

test_acc = (
    (test_preds == y_test.cpu())
    .sum()
    .item()
    / len(y_test)
)

print(f"Final Test Accuracy: {test_acc:.4f}")

Shape of concatenated embeddings: torch.Size([2449029, 128])


/tmp/ipykernel_57/2421032248.py:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx).to(device)
/tmp/ipykernel_57/2421032248.py:12: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid_idx = torch.tensor(valid_idx).to(device)
/tmp/ipykernel_57/2421032248.py:13: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx  = torch.tensor(test_idx).to(device)
/tmp/ipykernel_57/2421032248.py:14: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detac

cuda:0
cuda:0
Train nodes: 196615 | Valid nodes: 39323 | Test nodes: 2213091
Epoch   1/200 | Loss: 0.2875 | Train Acc: 0.9979 | Val Acc: 0.8166 | Gap: 0.1813
Epoch   2/200 | Loss: 0.0192 | Train Acc: 0.9987 | Val Acc: 0.8189 | Gap: 0.1799
Epoch   3/200 | Loss: 0.0183 | Train Acc: 0.9993 | Val Acc: 0.8093 | Gap: 0.1899
Epoch   4/200 | Loss: 0.0087 | Train Acc: 0.9996 | Val Acc: 0.8116 | Gap: 0.1880
Epoch   5/200 | Loss: 0.0048 | Train Acc: 0.9997 | Val Acc: 0.8051 | Gap: 0.1946
Epoch   6/200 | Loss: 0.0036 | Train Acc: 0.9998 | Val Acc: 0.8049 | Gap: 0.1949
Epoch   7/200 | Loss: 0.0029 | Train Acc: 0.9999 | Val Acc: 0.8023 | Gap: 0.1976
Epoch   8/200 | Loss: 0.0030 | Train Acc: 0.9999 | Val Acc: 0.7994 | Gap: 0.2004
Epoch   9/200 | Loss: 0.0028 | Train Acc: 0.9999 | Val Acc: 0.7963 | Gap: 0.2037
Epoch  10/200 | Loss: 0.0024 | Train Acc: 0.9999 | Val Acc: 0.8003 | Gap: 0.1996
Epoch  11/200 | Loss: 0.0026 | Train Acc: 0.9999 | Val Acc: 0.7993 | Gap: 0.2006


KeyboardInterrupt: 

In [14]:
S_generated = clip_fused

In [15]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score, f1_score



train_idx = torch.tensor(train_idx).to(device)
valid_idx =  torch.tensor(valid_idx).to(device)
test_idx  =  torch.tensor(test_idx).to(device)
labels_tensor = torch.tensor(labels_tensor).to(device) 

S_generated = S_generated.to(device)

S_train = S_generated[train_idx].detach()
y_train = labels_tensor[train_idx].long()

S_valid = S_generated[valid_idx].detach()
y_valid = labels_tensor[valid_idx].long()

S_test  = S_generated[test_idx].detach()
y_test  = labels_tensor[test_idx].long()



print("\n--- Transferring Data to CPU for Scikit-Learn ---")
X_train_np = S_train.cpu().numpy()
y_train_np = y_train.cpu().numpy()

X_test_np  = S_test.cpu().numpy()
y_test_np  = y_test.cpu().numpy()

print("Training Logistic Regression...")
clf = LogisticRegression(max_iter=1000, multi_class='multinomial', n_jobs=-1)
clf.fit(X_train_np, y_train_np)

print("Evaluating on Test Set...")
y_pred = clf.predict(X_test_np)


print(f"Accuracy: {accuracy_score(y_pred,y_test_np )}")


print("\n--- Logistic Regression Classification Report ---")
print(classification_report(y_test_np, y_pred, digits=4))

/tmp/ipykernel_57/3732168673.py:6: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  train_idx = torch.tensor(train_idx).to(device)
/tmp/ipykernel_57/3732168673.py:7: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  valid_idx =  torch.tensor(valid_idx).to(device)
/tmp/ipykernel_57/3732168673.py:8: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  test_idx  =  torch.tensor(test_idx).to(device)
/tmp/ipykernel_57/3732168673.py:9: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach(


--- Transferring Data to CPU for Scikit-Learn ---
Training Logistic Regression...


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:1247: FutureWarning: 'multi_class' was deprecated in version 1.5 and will be removed in 1.7. From then on, it will always use 'multinomial'. Leave it to its default value to avoid this warning.
  warnings.warn(


Evaluating on Test Set...
Accuracy: 0.7632804073578537

--- Logistic Regression Classification Report ---


/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

           0     0.6772    0.7951    0.7314    100670
           1     0.6954    0.7543    0.7237     95236
           2     0.8932    0.8344    0.8628    102870
           3     0.7560    0.6189    0.6806    135255
           4     0.9289    0.8800    0.9038    596195
           5     0.6635    0.8536    0.7466     35985
           6     0.7338    0.7904    0.7611    140356
           7     0.7758    0.9588    0.8577    152202
           8     0.7841    0.9614    0.8638     97679
           9     0.8736    0.8919    0.8826     59852
          10     0.6468    0.8230    0.7243     47165
          11     0.3140    0.4978    0.3851     28933
          12     0.8487    0.6206    0.7170    128632
          13     0.7508    0.7800    0.7651     88694
          14     0.3449    0.4682    0.3972      2734
          15     0.6962    0.9353    0.7982     23871
          16     0.6737    0.6600    0.6668     83019
          17     0.8497    

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
